In [14]:
import os, textwrap

print("\n".join(os.listdir("/kaggle/input/datasets/rohitrobertc2/galaxyeye-change-detection-v2")))

change_detection_assignment_data
train


In [2]:
# ── Cell 1 — Imports + Config ─────────────────────────────────────────────────

import os
import random
import numpy as np
import torch
import torch.nn as nn
import tifffile
import albumentations as A
import torchvision.models as tvm

from pathlib import Path
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.models import ResNet34_Weights

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Devise:", DEVICE)
device = DEVICE    # alias for compatibility

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR = Path("/kaggle/input/datasets/rohitrobertc2/galaxyeye-change-detection-v2")

PATHS = {
    "train": {
        "pre":    BASE_DIR / "train" / "pre-event",
        "post":   BASE_DIR / "train" / "post-event",
        "target": BASE_DIR / "train" / "target",
    },
    "val": {
        "pre":    BASE_DIR / "change_detection_assignment_data" / "val" / "pre-event",
        "post":   BASE_DIR / "change_detection_assignment_data" / "val" / "post-event",
        "target": BASE_DIR / "change_detection_assignment_data" / "val" / "target",
    },
    "test": {
        "pre":    BASE_DIR / "change_detection_assignment_data" / "test" / "pre-event",
        "post":   BASE_DIR / "change_detection_assignment_data" / "test" / "post-event",
        "target": BASE_DIR / "change_detection_assignment_data" / "test" / "target",
    },
}

# ── File Discovery ────────────────────────────────────────────────────────────
FILES = {}
for split in ["train", "val", "test"]:
    FILES[split] = sorted([p.name for p in PATHS[split]["pre"].glob("*.tif")])

for split in ["train", "val", "test"]:
    pre_n    = len(FILES[split])
    post_n   = len(list(PATHS[split]["post"].glob("*.tif")))
    target_n = len(list(PATHS[split]["target"].glob("*.tif")))
    print(f"{split}: pre={pre_n}, post={post_n}, target={target_n}")
    print(f"  sample: {FILES[split][0]}")

# ── Hyperparameters ───────────────────────────────────────────────────────────
IN_CHANNELS   = 9
NUM_CLASSES   = 1
BATCH_SIZE    = 8
NUM_EPOCHS    = 60
ENCODER_LR    = 1e-5
DECODER_LR    = 1e-3
WEIGHT_DECAY  = 3e-4
T_MAX         = 50
ETA_MIN       = 1e-6
PATIENCE      = 10
POS_WEIGHT    = 3.0
NUM_WORKERS   = 2
CROP_SIZE   = 1024
CHECKPOINT    = "best_v6.pth"

# Loss params
FOCAL_ALPHA   = 0.3
FOCAL_BETA    = 0.7
FOCAL_GAMMA   = 1.33
FOCAL_SMOOTH  = 1.0
LOVASZ_WEIGHT = 0.4

print("\nAll config loaded ✓")
print(f"Channels: {IN_CHANNELS} | Batch: {BATCH_SIZE} | Epochs: {NUM_EPOCHS}")

Devise: cuda
train: pre=2781, post=2781, target=2781
  sample: scene_01_000001_building_damage.tif
val: pre=334, post=334, target=334
  sample: scene_01_000001_building_damage.tif
test: pre=77, post=77, target=77
  sample: scene_09_000001_building_damage.tif

All config loaded ✓
Channels: 9 | Batch: 8 | Epochs: 60


In [3]:
# ── Cell 2 — ChangeDataset ────────────────────────────────────────────────────

class ChangeDataset(Dataset):
    def __init__(self, split, paths, file_names, transform=None):
        self.split      = split
        self.paths      = paths
        self.file_names = file_names
        self.transform  = transform

    def __len__(self):
        return len(self.file_names)

    # ── Step 1: Load raw files from disk ──────────────────────────────────────
    def _load_triplet(self, idx):
        fname = self.file_names[idx]

        pre  = tifffile.imread(self.paths[self.split]["pre"]    / fname)
        post = tifffile.imread(self.paths[self.split]["post"]   / fname)
        mask = tifffile.imread(self.paths[self.split]["target"] / fname)

        # SAR comes in as (1024,1024) — add channel dim → (1024,1024,1)
        post = post[:, :, np.newaxis]

        # Remap mask: 0=no change, 1=change
        # Raw values: 0=background, 1=uncertain, 2+=change
        mask = np.where(mask >= 2, 1, 0).astype(np.uint8)

        # ── EO-only brightness augmentation (training only) ───────────────────
        # Forces model to learn cross-modal relationships, not absolute EO values
        # SAR channels are NOT touched — backscatter values are physically meaningful
        if self.split == "train":
             factor = np.random.uniform(0.75, 1.25)
             pre = np.clip(pre.astype(np.float32) * factor, 0, 255).astype(np.uint8)
    # ──────────────────────────────────────────────────────────────────────
        return pre, post, mask, fname

        if self.split == "train":
            # EO brightness (existing)
            factor = np.random.uniform(0.75, 1.25)
            pre = np.clip(pre.astype(np.float32) * factor, 0, 255).astype(np.uint8)
    
           # SAR speckle simulation — multiplicative noise
           # Real SAR speckle is multiplicative, not additive
           # This forces the model to be robust to different SAR acquisition conditions
            noise = np.random.uniform(0.80, 1.20, size=post.shape).astype(np.float32)
            post = np.clip(post.astype(np.float32) * noise, 1, 231).astype(np.uint8) 

        return pre, post, mask, fname

    # ── Step 2: Engineer 7 channels ───────────────────────────────────────────
    def _engineer_features(self, pre, post):

        # Normalise FIRST — never compute on raw uint8
        pre_norm  = pre.astype(np.float32) / 255.0             # (H,W,3) [0,1]
        post_norm = (post.astype(np.float32) - 1.0) / 230.0   # (H,W,1) [0,1]
        post_norm = np.clip(post_norm, 0.0, 1.0)

        # EO Luminance — perceptual brightness of optical image
        eo_lum = (
            0.299 * pre_norm[:, :, 0:1] +
            0.587 * pre_norm[:, :, 1:2] +
            0.114 * pre_norm[:, :, 2:3]
        )                                                       # (H,W,1) [0,1]

        # Cross diff — signed disagreement between sensors
        # Positive → SAR bright, EO dim  → rubble
        # Negative → SAR dark,  EO medium → flood
        # Near zero → sensors agree       → no change
        cross     = post_norm - eo_lum                         # (H,W,1) [-1,1]
        cross_abs = np.abs(cross)                              # (H,W,1) [0,1]

        # ── Channel 7 — NGRDI (Normalized Green-Red Difference Index) ─────────
        # Proxy for vegetation health using RGB (no NIR available)
        # Destroyed vegetation and flooded land both drop NGRDI dramatically
        # Green = channel 1, Red = channel 0
        ngrdi = (
            (pre_norm[:,:,1:2] - pre_norm[:,:,0:1]) /
            (pre_norm[:,:,1:2] + pre_norm[:,:,0:1] + 1e-8)
        )                                                     # (H,W,1) [-1,1]

        # ── Channel 8 — SAR-NGRDI interaction ────────────────────────────────
        # V4 BUG FIX: normalise NGRDI to [0,1] BEFORE computing interaction
        # np.clip(-1,0) was silently zeroing all negative values (water, rubble)
        # making this channel a copy of post_norm for exactly those pixels
        ngrdi_norm = (ngrdi + 1.0) / 2.0                     # (H,W,1) [0,1]
        sar_ngrdi  = post_norm * (1.0 - ngrdi_norm)  # (H,W,1) [0,1]
        # ngrdi=-1 (destroyed vegetation) → ngrdi_norm=0 → sar_ngrdi=SAR*1.0 (amplified)
        # ngrdi=+1 (intact vegetation)    → ngrdi_norm=1 → sar_ngrdi=SAR*0.0 (suppressed)

        # Stack all  channels along last axis
        combined = np.concatenate(
            [pre_norm, post_norm, eo_lum, cross, cross_abs, ngrdi_norm, sar_ngrdi],
            axis=-1
        ).astype(np.float32)                                   # (H,W,7)

        return combined

    # ── Step 3: Augment + convert to tensor ───────────────────────────────────
    def __getitem__(self, idx):
        pre, post, mask, fname = self._load_triplet(idx)

        image = self._engineer_features(pre, post)             # (H,W,7)

        # Albumentations expects HWC for image, HW for mask
        if self.transform is not None:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]                         # still (H,W,7)
            mask  = augmented["mask"]                          # still (H,W)

        # PyTorch wants CHW — transpose from HWC
        image = torch.from_numpy(
            np.transpose(image, (2, 0, 1))
        ).float()                                              # (7,H,W)

        mask = torch.from_numpy(mask).float()                  # (H,W)

        return image, mask, fname


# ── Augmentation pipeline ─────────────────────────────────────────────────────
# Only geometric flips — NEVER colour augmentation on SAR channels
train_transform = A.Compose([
    A.RandomCrop(CROP_SIZE, CROP_SIZE, p=1.0),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
], is_check_shapes=False)                # Hard rule #10 — always include this

# Val and test — centre crop only, no augmentation
val_test_transform = A.Compose([
    A.CenterCrop(CROP_SIZE, CROP_SIZE, p=1.0),
], is_check_shapes=False)

# ── Sanity check — scene_09 raw mask values BEFORE remapping ─────────────────
print("=== scene_09 mask sanity check ===")
sample_test_mask_path = PATHS["test"]["target"] / FILES["test"][0]
raw_mask = tifffile.imread(sample_test_mask_path)

print(f"File: {FILES['test'][0]}")
print(f"Shape: {raw_mask.shape}")
print(f"Dtype: {raw_mask.dtype}")
print(f"Unique raw values: {np.unique(raw_mask)}")
print(f"Raw value counts:")
vals, counts = np.unique(raw_mask, return_counts=True)
for v, c in zip(vals, counts):
    print(f"  value {v}: {c} pixels ({100*c/raw_mask.size:.2f}%)")

remapped = np.where(raw_mask >= 2, 1, 0).astype(np.uint8)
print(f"\nAfter remap unique values: {np.unique(remapped)}")
print(f"Change pixels after remap: {remapped.sum()} ({100*remapped.mean():.2f}%)")
print("==================================")

=== scene_09 mask sanity check ===
File: scene_09_000001_building_damage.tif
Shape: (1024, 1024)
Dtype: uint8
Unique raw values: [0 1]
Raw value counts:
  value 0: 1020204 pixels (97.29%)
  value 1: 28372 pixels (2.71%)

After remap unique values: [0]
Change pixels after remap: 0 (0.00%)


In [4]:
# ── Cell 3 — Sampler ──────────────────────────────────────────────────────────

def compute_weights(target_dir, file_names, pos_weight=POS_WEIGHT):
    weights = []
    for fname in file_names:
        mask = tifffile.imread(target_dir / fname)
        mask = np.where(mask >= 2, 1, 0).astype(np.uint8)
        weights.append(pos_weight if np.any(mask == 1) else 1.0)
    return np.array(weights, dtype=np.float32)


train_weights = compute_weights(PATHS["train"]["target"], FILES["train"])

num_pos = int(np.sum(train_weights > 1.0))
num_neg = int(np.sum(train_weights == 1.0))
print(f"Tiles with change:    {num_pos}")
print(f"Tiles without change: {num_neg}")
print(f"Pos weight applied:   {POS_WEIGHT}x")

train_sampler = WeightedRandomSampler(
    weights=torch.tensor(train_weights, dtype=torch.float32),
    num_samples=len(train_weights),
    replacement=True
)

print("Sampler ready ✓")

Tiles with change:    713
Tiles without change: 2068
Pos weight applied:   3.0x
Sampler ready ✓


In [5]:
# ── Cell 4 — DataLoaders ──────────────────────────────────────────────────────

train_dataset = ChangeDataset(
    split="train",
    paths=PATHS,
    file_names=FILES["train"],
    transform=train_transform
)

val_dataset = ChangeDataset(
    split="val",
    paths=PATHS,
    file_names=FILES["val"],
    transform=val_test_transform
)

test_dataset = ChangeDataset(
    split="test",
    paths=PATHS,
    file_names=FILES["test"],
    transform=val_test_transform
)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# ── DataLoaders ───────────────────────────────────────────────────────────────
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

# ── Batch sanity check ────────────────────────────────────────────────────────
images, masks, fnames = next(iter(train_loader))
print(f"\nBatch sanity check:")
print(f"Images shape: {images.shape}")    # expect (8, 7, 1024, 1024)
print(f"Masks shape:  {masks.shape}")     # expect (8, 1024, 1024)
print(f"Images range: [{images.min():.3f}, {images.max():.3f}]")
print(f"Masks unique: {torch.unique(masks)}")
print(f"Sample file:  {fnames[0]}")

Train: 2781 | Val: 334 | Test: 77
Train batches: 348
Val batches:   42
Test batches:  10

Batch sanity check:
Images shape: torch.Size([8, 9, 1024, 1024])
Masks shape:  torch.Size([8, 1024, 1024])
Images range: [-1.000, 1.000]
Masks unique: tensor([0., 1.])
Sample file:  scene_02_000080_building_damage.tif


In [6]:
# ── Cell 5 — Model ────────────────────────────────────────────────────────────

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.15):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=dropout),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=dropout),
        )

    def forward(self, x):
        return self.block(x)


class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
        self.conv = ConvBlock(in_ch // 2 + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class ResNet34UNet(nn.Module):
    def __init__(self, in_channels=IN_CHANNELS, num_classes=NUM_CLASSES):
        super().__init__()

        enc = tvm.resnet34(weights=ResNet34_Weights.DEFAULT)

        # ── Patch conv1 to accept 7 channels instead of 3 ────────────────────
        old_w   = enc.conv1.weight.data                        # (64, 3, 7, 7)
        new_conv = nn.Conv2d(
            in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False
        )
        with torch.no_grad():
            repeated = old_w.repeat(1, in_channels // 3 + 1, 1, 1)[:, :in_channels]
            new_conv.weight.data = repeated / (in_channels / 3)
        enc.conv1 = new_conv

        # ── Encoder stages ────────────────────────────────────────────────────
        self.e0   = nn.Sequential(enc.conv1, enc.bn1, enc.relu)  # (B, 64,  H/2,  W/2)
        self.pool = enc.maxpool
        self.e1   = enc.layer1                                    # (B, 64,  H/4,  W/4)
        self.e2   = enc.layer2                                    # (B, 128, H/8,  W/8)
        self.e3   = enc.layer3                                    # (B, 256, H/16, W/16)
        self.e4   = enc.layer4                                    # (B, 512, H/32, W/32)

        # ── Decoder stages ────────────────────────────────────────────────────
        self.d3 = UpBlock(512, 256, 256)
        self.d2 = UpBlock(256, 128, 128)
        self.d1 = UpBlock(128, 64,  64)
        self.d0 = UpBlock(64,  64,  32)

        self.up_final = nn.ConvTranspose2d(32, 32, kernel_size=2, stride=2)
        self.head     = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, x):
        # Encoder — progressively compress spatial size, expand features
        s0 = self.e0(x)
        s1 = self.e1(self.pool(s0))
        s2 = self.e2(s1)
        s3 = self.e3(s2)
        b  = self.e4(s3)

        # Decoder — upsample back, skip connections restore spatial detail
        x = self.d3(b,  s3)
        x = self.d2(x,  s2)
        x = self.d1(x,  s1)
        x = self.d0(x,  s0)

        x = self.up_final(x)
        return self.head(x)                                    # raw logits (B,1,H,W)


# ── Instantiate + shape verification ─────────────────────────────────────────
model = ResNet34UNet(in_channels=IN_CHANNELS, num_classes=NUM_CLASSES).to(DEVICE)

model.eval()
with torch.no_grad():
    dummy = torch.zeros(1, IN_CHANNELS, 1024, 1024).to(DEVICE)
    out   = model(dummy)

print(f"Input shape:  {dummy.shape}")     # (1, 7, 1024, 1024)
print(f"Output shape: {out.shape}")       # (1, 1, 1024, 1024)
print(f"Output range: [{out.min():.3f}, {out.max():.3f}]")
print("Model ready ✓")

try:
    enc = tvm.resnet34(weights=ResNet34_Weights.DEFAULT)
    print("✓ Loaded pre-trained weights")
except Exception as e:
    print(f"⚠ Could not download weights: {e}")
    print("Using random initialization instead")
    enc = tvm.resnet34(weights=None)

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 170MB/s] 


Input shape:  torch.Size([1, 9, 1024, 1024])
Output shape: torch.Size([1, 1, 1024, 1024])
Output range: [-0.241, -0.222]
Model ready ✓
✓ Loaded pre-trained weights


In [7]:
# ── Cell 6 — FocalTverskyLoss ─────────────────────────────────────────────────

class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=FOCAL_ALPHA, beta=FOCAL_BETA,
                 gamma=FOCAL_GAMMA, smooth=FOCAL_SMOOTH):
        super().__init__()
        self.alpha  = alpha
        self.beta   = beta
        self.gamma  = gamma
        self.smooth = smooth

    def forward(self, pred, target):
        pred   = torch.sigmoid(pred)
        pred   = pred.view(-1)
        target = target.view(-1).float()

        tp = (pred * target).sum()
        fp = ((1 - target) * pred).sum()
        fn = (target * (1 - pred)).sum()

        tversky = (tp + self.smooth) / (
            tp + self.alpha * fp + self.beta * fn + self.smooth
        )

        return (1 - tversky) ** self.gamma


loss_fn = FocalTverskyLoss()

# ── Sanity checks ─────────────────────────────────────────────────────────────
logits_good_bg = torch.full((1, 1, 4, 4), -4.0)
logits_good_fg = torch.full((1, 1, 4, 4),  4.0)
mask_all_zeros = torch.zeros(1, 4, 4)
mask_all_ones  = torch.ones(1, 4, 4)

print("Loss (predict bg correctly):  ", loss_fn(logits_good_bg, mask_all_zeros).item())
print("Loss (predict change correctly):", loss_fn(logits_good_fg, mask_all_ones).item())

torch.manual_seed(SEED)
logits_mix = torch.randn(1, 1, 4, 4, requires_grad=True)
mask_mix   = torch.randint(0, 2, (1, 4, 4)).float()
loss_mix   = loss_fn(logits_mix, mask_mix)
loss_mix.backward()

print("Loss (mixed case):            ", loss_mix.item())
print("Gradients finite:             ", torch.isfinite(logits_mix.grad).all().item())
print(f"Gamma used: {FOCAL_GAMMA}  ← should be 1.33")
print("Loss ready ✓")

Loss (predict bg correctly):   0.03445792943239212
Loss (predict change correctly): 0.0027604205533862114
Loss (mixed case):             0.280682235956192
Gradients finite:              True
Gamma used: 1.33  ← should be 1.33
Loss ready ✓


In [8]:
# ── Cell 7 — Optimizer + Scheduler + Early Stopping ──────────────────────────

encoder_params = (
    list(model.e0.parameters()) +
    list(model.e1.parameters()) +
    list(model.e2.parameters()) +
    list(model.e3.parameters()) +
    list(model.e4.parameters())
)

decoder_params = (
    list(model.d3.parameters()) +
    list(model.d2.parameters()) +
    list(model.d1.parameters()) +
    list(model.d0.parameters()) +
    list(model.up_final.parameters()) +
    list(model.head.parameters())
)

optimizer = torch.optim.AdamW(
    [
        {"params": encoder_params, "lr": ENCODER_LR},
        {"params": decoder_params, "lr": DECODER_LR},
    ],
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=T_MAX,
    eta_min=ETA_MIN
)

# ── Early stopping state ──────────────────────────────────────────────────────
best_iou   = 0.0
no_improve = 0

print(f"Encoder LR:  {ENCODER_LR}")
print(f"Decoder LR:  {DECODER_LR}")
print(f"Weight decay: {WEIGHT_DECAY}")
print(f"Scheduler T_max: {T_MAX} | eta_min: {ETA_MIN}")
print(f"Early stopping patience: {PATIENCE}")
print("Optimizer ready ✓")

Encoder LR:  1e-05
Decoder LR:  0.001
Weight decay: 0.0003
Scheduler T_max: 50 | eta_min: 1e-06
Early stopping patience: 10
Optimizer ready ✓


In [9]:
# ── Cell 8 — Metrics ──────────────────────────────────────────────────────────

def compute_metrics(pred_logits, true_mask, threshold=0.4):
    pred   = (torch.sigmoid(pred_logits) > threshold).float().view(-1)
    target = true_mask.float().view(-1)

    eps = 1e-8

    tp = ((pred == 1) & (target == 1)).sum().float()
    fp = ((pred == 1) & (target == 0)).sum().float()
    fn = ((pred == 0) & (target == 1)).sum().float()
    tn = ((pred == 0) & (target == 0)).sum().float()

    iou       = tp / (tp + fp + fn + eps)
    precision = tp / (tp + fp + eps)
    recall    = tp / (tp + fn + eps)
    f1        = 2 * precision * recall / (precision + recall + eps)

    return {"tp": tp.item(), "fp": fp.item(),
            "fn": fn.item(), "tn": tn.item(),
            "iou": iou.item(), "precision": precision.item(),
            "recall": recall.item(), "f1": f1.item()}


def compute_epoch_metrics(tp, fp, fn, tn):
    eps = 1e-8
    tp  = torch.tensor(tp, dtype=torch.float32)
    fp  = torch.tensor(fp, dtype=torch.float32)
    fn  = torch.tensor(fn, dtype=torch.float32)
    tn  = torch.tensor(tn, dtype=torch.float32)

    iou       = tp / (tp + fp + fn + eps)
    precision = tp / (tp + fp + eps)
    recall    = tp / (tp + fn + eps)
    f1        = 2 * precision * recall / (precision + recall + eps)

    return {"iou": iou.item(), "precision": precision.item(),
            "recall": recall.item(), "f1": f1.item()}


# ── Sanity checks ─────────────────────────────────────────────────────────────
m_perfect = compute_metrics(
    torch.full((1, 1, 4, 4), 4.0),
    torch.ones(1, 4, 4)
)
m_wrong = compute_metrics(
    torch.full((1, 1, 4, 4), 4.0),
    torch.zeros(1, 4, 4)
)

print(f"Perfect prediction → IoU: {m_perfect['iou']:.4f} | F1: {m_perfect['f1']:.4f}")
print(f"Wrong prediction   → IoU: {m_wrong['iou']:.4f}  | F1: {m_wrong['f1']:.4f}")
print("Metrics ready ✓")

Perfect prediction → IoU: 1.0000 | F1: 1.0000
Wrong prediction   → IoU: 0.0000  | F1: 0.0000
Metrics ready ✓


In [10]:
# ── Cell 9 — Train One Epoch ──────────────────────────────────────────────────

def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()

    total_loss   = 0.0
    total_pixels = 0
    tp = fp = fn = tn = 0.0

    for images, masks, _ in loader:
        images = images.to(device)
        masks  = masks.to(device)

        optimizer.zero_grad()

        logits = model(images)                    # (B, 1, H, W)
        loss   = loss_fn(logits, masks)

        loss.backward()
        optimizer.step()

        # Accumulate loss weighted by pixel count
        n_pixels      = images.size(0) * masks.shape[-2] * masks.shape[-1]
        total_loss   += loss.item() * n_pixels
        total_pixels += n_pixels

        # Accumulate raw counts — never average per batch
        m   = compute_metrics(logits.detach(), masks.detach())
        tp += m["tp"];  fp += m["fp"]
        fn += m["fn"];  tn += m["tn"]

    return {
        "loss": total_loss / max(total_pixels, 1),
        "tp": tp, "fp": fp, "fn": fn, "tn": tn
    }

In [11]:
# ── Cell 10 — Evaluate One Epoch ─────────────────────────────────────────────

def evaluate_one_epoch(model, loader, loss_fn, device):
    model.eval()

    total_loss   = 0.0
    total_pixels = 0
    tp = fp = fn = tn = 0.0

    with torch.no_grad():
        for images, masks, _ in loader:
            images = images.to(device)
            masks  = masks.to(device)

            logits = model(images)
            loss   = loss_fn(logits, masks)

            n_pixels      = images.size(0) * masks.shape[-2] * masks.shape[-1]
            total_loss   += loss.item() * n_pixels
            total_pixels += n_pixels

            m   = compute_metrics(logits, masks)
            tp += m["tp"];  fp += m["fp"]
            fn += m["fn"];  tn += m["tn"]

    return {
        "loss": total_loss / max(total_pixels, 1),
        "tp": tp, "fp": fp, "fn": fn, "tn": tn
    }

In [ ]:
# ── Cell 11 — Training Driver ─────────────────────────────────────────────────

train_history = []
val_history   = []


for epoch in range(1, NUM_EPOCHS + 1):

    train_stats   = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
    val_stats     = evaluate_one_epoch(model, val_loader, loss_fn, device)

    train_metrics = compute_epoch_metrics(
        train_stats["tp"], train_stats["fp"],
        train_stats["fn"], train_stats["tn"]
    )
    val_metrics   = compute_epoch_metrics(
        val_stats["tp"], val_stats["fp"],
        val_stats["fn"], val_stats["tn"]
    )

    scheduler.step()

    # ── Logging ───────────────────────────────────────────────────────────────
    train_history.append({"loss": train_stats["loss"], **train_metrics})
    val_history.append({"loss": val_stats["loss"],     **val_metrics})

    current_lrs = [g["lr"] for g in optimizer.param_groups]

    print(
        f"Epoch {epoch:03d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_stats['loss']:.4f} | "
        f"Val Loss: {val_stats['loss']:.4f} | "
        f"Train IoU: {train_metrics['iou']:.4f} | "
        f"Val IoU: {val_metrics['iou']:.4f} | "
        f"LR enc: {current_lrs[0]:.2e} dec: {current_lrs[1]:.2e}"
    )

    # ── Checkpoint ────────────────────────────────────────────────────────────
    val_iou   = val_metrics["iou"]
    improved  = val_iou > best_iou + 1e-5

    if improved:
        best_iou   = val_iou
        no_improve = 0
        torch.save(model.state_dict(), CHECKPOINT)
        print(f"  ✓ Saved best model (Val IoU: {best_iou:.4f})")
    else:
        no_improve += 1
        print(f"  No improvement {no_improve}/{PATIENCE}")

    # ── Early stopping ────────────────────────────────────────────────────────
    if no_improve >= PATIENCE:
        print(f"Early stopping at epoch {epoch}")
        break

print(f"\nTraining complete. Best Val IoU: {best_iou:.4f}")

In [12]:
# ── Cell 12 — Final Evaluation with TTA ───────────────────────────────────────

best_model = ResNet34UNet(in_channels=IN_CHANNELS, num_classes=NUM_CLASSES).to(DEVICE)
best_model.load_state_dict(
    torch.load(CHECKPOINT, map_location=DEVICE, weights_only=True)
)
best_model.eval()

def tta_evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss   = 0.0
    total_pixels = 0
    tp = fp = fn = tn = 0.0

    with torch.no_grad():
        for images, masks, _ in loader:
            images = images.to(device)
            masks  = masks.to(device)

            # Four forward passes — original + 3 flips
            pred_orig = torch.sigmoid(model(images))

            pred_hflip = torch.sigmoid(model(torch.flip(images, dims=[3])))
            pred_hflip = torch.flip(pred_hflip, dims=[3])

            pred_vflip = torch.sigmoid(model(torch.flip(images, dims=[2])))
            pred_vflip = torch.flip(pred_vflip, dims=[2])

            pred_hvflip = torch.sigmoid(model(torch.flip(images, dims=[2,3])))
            pred_hvflip = torch.flip(pred_hvflip, dims=[2,3])

            # Average all 4 predictions
            avg_pred = (pred_orig + pred_hflip + pred_vflip + pred_hvflip) / 4.0

            # Convert averaged probability to logit for loss
            avg_logit = torch.log(avg_pred / (1 - avg_pred + 1e-8) + 1e-8)

            loss = loss_fn(avg_logit, masks)

            n_pixels      = images.size(0) * masks.shape[-2] * masks.shape[-1]
            total_loss   += loss.item() * n_pixels
            total_pixels += n_pixels

            m   = compute_metrics(avg_logit, masks)
            tp += m["tp"];  fp += m["fp"]
            fn += m["fn"];  tn += m["tn"]

    return {
        "loss": total_loss / max(total_pixels, 1),
        "tp": tp, "fp": fp, "fn": fn, "tn": tn
    }

# ── Validation ────────────────────────────────────────────────────────────────
val_stats   = tta_evaluate(best_model, val_loader, loss_fn, DEVICE)
val_metrics = compute_epoch_metrics(
    val_stats["tp"], val_stats["fp"],
    val_stats["fn"], val_stats["tn"]
)

print("=" * 60)
print("VALIDATION RESULTS — best_v5.pth + TTA + threshold=0.4")
print("=" * 60)
print(f"Loss:      {val_stats['loss']:.4f}")
print(f"IoU:       {val_metrics['iou']:.4f}")
print(f"Precision: {val_metrics['precision']:.4f}")
print(f"Recall:    {val_metrics['recall']:.4f}")
print(f"F1:        {val_metrics['f1']:.4f}")
print(f"TP: {val_stats['tp']:.0f} | FP: {val_stats['fp']:.0f} | "
      f"FN: {val_stats['fn']:.0f} | TN: {val_stats['tn']:.0f}")

# ── Test ──────────────────────────────────────────────────────────────────────
test_stats   = tta_evaluate(best_model, test_loader, loss_fn, DEVICE)
test_metrics = compute_epoch_metrics(
    test_stats["tp"], test_stats["fp"],
    test_stats["fn"], test_stats["tn"]
)

print("=" * 60)
print("TEST RESULTS — best_v5.pth + TTA + threshold=0.4")
print("=" * 60)
print(f"Loss:      {test_stats['loss']:.4f}")
print(f"IoU:       {test_metrics['iou']:.4f}")
print(f"Precision: {test_metrics['precision']:.4f}")
print(f"Recall:    {test_metrics['recall']:.4f}")
print(f"F1:        {test_metrics['f1']:.4f}")
print(f"TP: {test_stats['tp']:.0f} | FP: {test_stats['fp']:.0f} | "
      f"FN: {test_stats['fn']:.0f} | TN: {test_stats['tn']:.0f}")

FileNotFoundError: [Errno 2] No such file or directory: 'best_v6.pth'

In [29]:
from IPython.display import FileLink
FileLink('best_v5.pth')

/kaggle/working/best_v5.pth